# Hit-Rate Analysis
Compare success rate (_hit rate_) and sensitivity (_d'_ or _A'_) across different stimulus features. In other words, we check $P[miss]$ across the following conditions:
- **Target Category** is the category of the target object (`animal body`/`animal face`/`human body`/`human face`/`inanimate handmade`/`inanimate natural`)
- **Trial Category** is the presentation mode of the search array (`color`/`grayscale`/`noisy background`)
- **target Rotation** is the (absolute) rotation angle of the target (`0°`,`1°`, ..., `20°`)

In [1]:
import os

from analysis.helpers.r_bridge import setup_rpy2, to_r_dataframe, source_r, get_r_object, glmer_metrics, cached_fit
setup_rpy2()

import polars as pl
import pandas as pd
from pymer4.models import glmer
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

import config as cnfg
import pipeline.config as pcfg
from analysis.helpers.read_data import load_data, parse_as_categorical
from analysis.helpers.constants import TRIAL_SYMBOLS, TARGET_SYMBOLS
from analysis.helpers.aggregation import calc_miss_rate
from pipeline.stage3_classify.trial_inclusion import check_trial_inclusion_criteria
from utils.sdt import calc_sdt_metrics
from data_models.LWSEnums import SearchArrayCategoryEnum, ImageCategoryEnum

pio.renderers.default = "browser"      # or "browser"

## Prepare Data
### Read Data

In [2]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
targets = loaded_data.targets
actions = loaded_data.actions
metadata = loaded_data.metadata
idents = loaded_data.identifications
fixations = loaded_data.fixations
del loaded_data

### Merge Identifications with Trial & Target Metadata

In [3]:
idents = (
    idents
    .merge(
        metadata[["subject", "trial", "trial_category"]], on=["subject", "trial"], how="left"
    )
    .merge(
        targets[["subject", "trial", "target", "category", "angle"]],
        on=["subject", "trial", "target"],
        how="left",
    )
    .rename(columns={"category": "target_category", "angle": "target_angle"})
    .astype({"subject": "category", "trial": int, "target": "category", "target_angle": float,})
    .assign(
        is_hit=lambda df: (df["identification_category"] == "hit").astype(bool),
        abs_target_angle=lambda df: df["target_angle"].abs().astype(float),
        trial_category=lambda df: parse_as_categorical(
            df["trial_category"], SearchArrayCategoryEnum, True
        ),
        target_category=lambda df: parse_as_categorical(
            df["target_category"], ImageCategoryEnum, False
        ),
    )
    .sort_values(["subject", "trial", "target"])
    .reset_index(drop=True)
)

### Calculate SDT Metric per Trial

In [4]:
trial_sdt_scores = calc_sdt_metrics(metadata, idents, "loglinear")

### Exclude Bad Trials

In [5]:
inclusion_criteria = check_trial_inclusion_criteria(
    metadata, fixations, actions, idents,
    min_gaze_coverage=pcfg.DEFAULT_GAZE_COVERAGE_PERCENT_THRESHOLD,
    min_fixation_rate=pcfg.DEFAULT_FIXATION_RATE_THRESHOLD,
    bad_actions=pcfg.DEFAULT_BAD_ACTIONS,
    require_actions=False,
)

In [6]:
idents = idents.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
idents = idents.loc[idents["is_valid_trial"]]      # apply trial filter

metadata = metadata.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
metadata = metadata.loc[metadata["is_valid_trial"]]      # apply trial filter

trial_sdt_scores = trial_sdt_scores.merge(
    inclusion_criteria["is_valid_trial"].reset_index(), on=["subject", "trial"], how="left"
)
trial_sdt_scores = trial_sdt_scores.loc[trial_sdt_scores["is_valid_trial"]]

### Calculate SDT per Trial Category

In [7]:
scores_per_subject = (
    trial_sdt_scores
    .groupby("subject")
    .agg(
        n_trials=("trial_category", "count"),
        hit_rate=("hit_rate", "mean"),
        hit_rate_std=("hit_rate", "std"),
        fa_rate=("false_alarm_rate", "mean"),
        fa_rate_std=("false_alarm_rate", "std"),
        d_prime=("d_prime", "mean"),
        d_prime_std=("d_prime", "std"),
        a_prime=("a_prime", "mean"),
        a_prime_std=("a_prime", "std"),
        f1_score=("f1_score", "mean"),
        f1_score_std=("f1_score", "std"),
    )
    .assign(trial_category="all")
    .reset_index(drop=False)
)
scores_per_trial_category = (
    trial_sdt_scores
    .groupby(["subject", "trial_category"])
    .agg(
        n_trials=("trial_category", "count"),
        hit_rate=("hit_rate", "mean"),
        hit_rate_sem=("hit_rate", "sem"),
        fa_rate=("false_alarm_rate", "mean"),
        fa_rate_sem=("false_alarm_rate", "sem"),
        d_prime=("d_prime", "mean"),
        d_prime_sem=("d_prime", "sem"),
        a_prime=("a_prime", "mean"),
        a_prime_sem=("a_prime", "sem"),
        f1_score=("f1_score", "mean"),
        f1_score_sem=("f1_score", "sem"),
    )
    .reset_index(drop=False)
)
scores_per_trial_category = (
    pd.concat([scores_per_trial_category, scores_per_subject], ignore_index=True)
    .assign(
        trial_category=lambda df: parse_as_categorical(df["trial_category"], SearchArrayCategoryEnum, True),
    )
    .dropna(subset=["n_trials"])
    .sort_values(["subject", "trial_category"])
    .reset_index(drop=True)
)

## Descriptive Statistics
We calculate the percentage of _MISSED_ targets for each subject, and drill down to look at the statistics across trial types and target categories

In [8]:
subjects_miss_rate = calc_miss_rate(idents, ["subject"])
average_miss_rate = subjects_miss_rate["miss_rate"].agg(["count", "mean", "sem"])
print(f"Average Miss-Rate across {average_miss_rate['count']} Subjects:\t{100 * average_miss_rate['mean'] :.2f} ± {100 * average_miss_rate['sem'] :.2f}%")

print(f"Average Miss-Rates across Trial Types:")
trialtype_miss_rate = (
    calc_miss_rate(idents, ["subject", "trial_category"])
    .groupby("trial_category", observed=True)["miss_rate"]
    .agg(["mean", "sem"])
)
display(trialtype_miss_rate)

print(f"Average Miss-Rates across Target Categories:")
targetcat_miss_rate = (
    calc_miss_rate(idents, ["subject", "target_category"])
    .groupby("target_category", observed=True)["miss_rate"]
    .agg(["mean", "sem"])
)
display(targetcat_miss_rate)

Average Miss-Rate across 28.0 Subjects:	25.01 ± 1.57%
Average Miss-Rates across Trial Types:


,mean,sem
trial_category,,
COLOR,0.194204,0.020121
BW,0.350774,0.025008
NOISE,0.201866,0.016668


Average Miss-Rates across Target Categories:


,mean,sem
target_category,,
ANIMAL_OTHER,0.326951,0.029705
HUMAN_OTHER,0.260951,0.019416
ANIMAL_FACE,0.363346,0.027413
HUMAN_FACE,0.186559,0.026620
OBJECT_HANDMADE,0.177036,0.020251
OBJECT_NATURAL,0.186475,0.026014


## Descriptive Figures
### SDT Measures by Trial Category

In [9]:
metrics = {"hit_rate": "Hit Rate", "f1_score": "$f1$ Score", "a_prime": "$A'$"}

fig = make_subplots(
    rows=len(metrics), cols=2, column_widths=[0.8, 0.2,],
    shared_yaxes=True, shared_xaxes=True, x_title="Trial Category",
    vertical_spacing=0.075, horizontal_spacing=0.025,
    subplot_titles=[ttl for tup in zip(metrics.values(), [""] * len(metrics)) for ttl in tup]
)
for r, (metric, metric_name) in enumerate(metrics.items()):
    for c, is_aggregated in enumerate([False, True]):
        if is_aggregated:
            data = scores_per_trial_category[scores_per_trial_category["trial_category"] == "all"]
        else:
            data = scores_per_trial_category[scores_per_trial_category["trial_category"] != "all"]
        # add distribution plots
        fig.add_trace(
            row=r+1, col=c+1, trace=go.Violin(
                x=data["trial_category"], y=data[metric],
                name=f"{metric_name} Distribution",
                marker=dict(color="gray", line=dict(color="black", width=1)),
                opacity=0.75, points=False, spanmode="hard", width=0.5,
                box=dict(visible=False), meanline=dict(visible=True),
                showlegend=False,
            ))
        # add individual subject points
        for i, subj in enumerate(data["subject"].unique()):
            subj_data = data[data["subject"] == subj]
            subj_name = f"Subject {subj}"
            subj_color = cnfg.get_discrete_color(i, loop=True)
            subj_color = (int(subj_color[1:3], 16), int(subj_color[3:5], 16), int(subj_color[5:7], 16))
            fig.add_trace(
                row=r+1, col=c+1, trace=go.Scatter(
                    x=subj_data["trial_category"], y=subj_data[metric],
                    error_y=dict(
                        type="data", array=subj_data[f"{metric}_std"],
                        color=f"rgba{subj_color + (1,)}", thickness=1.5, width=3,
                        visible=is_aggregated,
                    ),
                    name=subj_name, legendgroup=subj_name, showlegend=(r==0 and c==0),
                    mode="markers+lines",
                    marker=dict(size=6, color=f"rgba{subj_color + (1,)}"),
                    line=dict(width=2, color=f"rgba{subj_color + (0.5,)}"),
                    text=f"<b>Subject</b>:\t{subj}", hoverinfo="text+y",
            ))

fig.update_layout(
    width=1000, height=1000,
    title=dict(text="Behavioral Measures across Trial Types"),
)
fig.show()

### Hit-Rate per Target Category

In [10]:
grouped = []
for groupers in [["subject"], ["subject", "target_category"]]:
    hit_rate = (
        idents
        .loc[idents["identification_category"] != "false_alarm",]
        .groupby(groupers, observed=False)
        .agg(
            n_idents=("identification_category", "count"),
            n_hits=("identification_category", lambda x: (x == "hit").sum()),
        )
        .assign(hit_rate=lambda df: df["n_hits"] / df["n_idents"],)
        .dropna(subset=["hit_rate"])
        .reset_index(drop=False)
    )
    if "target_category" not in groupers:
        hit_rate["target_category"]  = "all"
    grouped.append(hit_rate)

hit_rate_stats = (
    pd.concat(grouped)
    .assign(
        target_category=lambda df: parse_as_categorical(df["target_category"], ImageCategoryEnum, False)
    )
    .sort_values(["subject", "target_category"])
    .reset_index(drop=True)
)

In [11]:
fig = make_subplots(
    rows=1, cols=2, column_widths=[0.8, 0.2,],
    shared_yaxes=True, shared_xaxes=True,
    subplot_titles=["Hit Rate", ""], x_title="Target Category",
    vertical_spacing=0.1, horizontal_spacing=0.05,
)
for c, is_aggregated in enumerate([False, True]):
    if is_aggregated:
        data = hit_rate_stats[hit_rate_stats["target_category"] == "all"]
    else:
        data = hit_rate_stats[hit_rate_stats["target_category"] != "all"]
    # add distribution plots
    fig.add_trace(
        row=1, col=c+1, trace=go.Violin(
            x=data["target_category"], y=data["hit_rate"],
            name="Hit Rate Distribution",
            marker=dict(color="gray", line=dict(color="black", width=1)),
            opacity=0.75, points=False, spanmode="hard", width=0.75,
            box=dict(visible=False), meanline=dict(visible=True),
            showlegend=False,
        ))
    # add individual subject points
    for i, subj in enumerate(data["subject"].unique()):
        subj_data = data[data["subject"] == subj]
        subj_name = f"Subject {subj}"
        subj_color = cnfg.get_discrete_color(i, loop=True)
        subj_color = (int(subj_color[1:3], 16), int(subj_color[3:5], 16), int(subj_color[5:7], 16))
        fig.add_trace(
            row=1, col=c+1, trace=go.Scatter(
                x=subj_data["target_category"], y=subj_data["hit_rate"],
                name=subj_name, legendgroup=subj_name, showlegend=(c==0),
                mode="markers+lines",
                marker=dict(size=6, color=f"rgba{subj_color + (1,)}"),
                line=dict(width=2, color=f"rgba{subj_color + (0.25,)}"),
                text=f"<b>Subject</b>:\t{subj}", hoverinfo="text+y",
        ))

fig.update_xaxes(
    row=1, col=1,
    tickmode="array",
    tickvals=hit_rate_stats["target_category"].unique(),
    ticktext=[cat.replace("_", "<br>") for cat in hit_rate_stats["target_category"].unique()]
)
for ann in fig.layout.annotations:
    if ann.text == "Target Category":
        ann.x=0.375
        ann.xanchor="center"
        ann.y = -0.01
fig.update_layout(
    width=900, height=600,
    title=dict(text="Hit Rate by Target Category"),
)
fig.show()

### Hit-Rate by (Absolute) Target Angle

In [12]:
grouped = []
for groupers in [["subject", "abs_target_angle"], ["subject", "abs_target_angle", "trial_category"]]:
    hit_rate = (
        idents
        .loc[idents["identification_category"] != "false_alarm",]
        .groupby(groupers, observed=False)
        .agg(
            n_idents=("identification_category", "count"),
            n_hits=("identification_category", lambda x: (x == "hit").sum()),
        )
        .assign(hit_rate=lambda df: df["n_hits"] / df["n_idents"],)
        .dropna(subset=["hit_rate"])
        .reset_index(drop=False)
    )
    if "trial_category" not in groupers:
        hit_rate["trial_category"]  = "all"
    grouped.append(hit_rate)

hit_rate_over_subjects = (
    pd.concat(grouped)
    .assign(
        trial_category=lambda df: parse_as_categorical(df["trial_category"], SearchArrayCategoryEnum, False)
    )
    .groupby(["abs_target_angle", "trial_category"], observed=False)   # aggregate over subjects
    .agg(
        N=("hit_rate", "count"),
        average_rate=("hit_rate", "mean"),
        sem_rate=("hit_rate", "sem"),
    )
    .sort_index()
    .reset_index(drop=False)
    .assign(
        color=lambda df: df['trial_category'].map(lambda cat: cnfg.get_discrete_color(
                df['trial_category'].unique().tolist().index(cat), loop=True
        )),
        symbol=lambda df: df['trial_category'].map(TRIAL_SYMBOLS)
    )
)

In [13]:
fig = go.Figure()
for category in hit_rate_over_subjects["trial_category"].unique():
    category_data = hit_rate_over_subjects[hit_rate_over_subjects["trial_category"] == category]
    fig.add_trace(go.Scatter(
        x=category_data["abs_target_angle"],
        y=category_data["average_rate"],
        error_y=dict(
            type="data",
            array=category_data["sem_rate"],
            visible=True,
        ),
        mode="markers",
        name=category,
        marker=dict(size=8, symbol=category_data["symbol"], color=category_data["color"],),
        line=dict(color=category_data["color"].iloc[0], width=2),
    ))

fig.update_layout(
    title=dict(
        text="Hit Rate by Absolute Target Angle<br>(averaged across subjects)",
        font=cnfg.TITLE_FONT
    ),
    xaxis=dict(title=dict(text="Absolute Target Rotation Angle (﷿﷿)")),
    yaxis=dict(title=dict(text="Hit Rate"), range=[-0.05, 1.05]),
)
fig.show()

## Bayesian Hierarchical Analysis
### Hit-Rate against Icon Features
We fit (using `pymer4` and _R_'s `lme` packages) the hierarchical Generalized Linear Model (hGLM):
$$ \text{logit}(p[hit]) \sim abs(angle) * C(trial\_category) * C(target\_category) + (1 | subject) $$

In [14]:
hits_for_angle_regression = idents.loc[idents["identification_category"] != "false_alarm",]
model = glmer(
    formula="is_hit ~ abs_target_angle * trial_category * target_category + (1 | subject)",
    data=pl.from_pandas(hits_for_angle_regression),
    family="binomial",      # logistic regression in R uses `Binomial` (with parameter `n=1`)
)
model.set_factors(["subject", "trial_category", "target_category"])
result = model.fit(exponentiate=True, summary=True)
print(model.convergence_status)

R messages: 
Derivatives (gradient and/or Hessian) not available. Cannot assess
  convergence through gradient-based checks.

Convergence status
: [1] FALSE
attr(,"gradient")
[1] NA

R messages: 
Derivatives (gradient and/or Hessian) not available. Cannot assess
  convergence through gradient-based checks.

Convergence status
: [1] FALSE
attr(,"gradient")
[1] NA

Convergence status
: [1] FALSE
attr(,"gradient")
[1] NA



In [15]:
model.anova()
model.summary_anova()

GT(_tbl_data=shape: (7, 7)
┌─────────────────────────────────┬──────┬─────┬─────────┬────────┬─────────┬───────┐
│ model term                      ┆ df1  ┆ df2 ┆ F_ratio ┆ Chisq  ┆ p_value ┆ stars │
│ ---                             ┆ ---  ┆ --- ┆ ---     ┆ ---    ┆ ---     ┆ ---   │
│ str                             ┆ f64  ┆ f64 ┆ f64     ┆ f64    ┆ str     ┆ str   │
╞═════════════════════════════════╪══════╪═════╪═════════╪════════╪═════════╪═══════╡
│ abs_target_angle                ┆ 1.0  ┆ inf ┆ 0.485   ┆ 0.485  ┆ 0.486   ┆       │
│ trial_category                  ┆ 2.0  ┆ inf ┆ 39.1    ┆ 78.2   ┆ <.001   ┆ ***   │
│ target_category                 ┆ 5.0  ┆ inf ┆ 17.967  ┆ 89.835 ┆ <.001   ┆ ***   │
│ abs_target_angle:trial_categor… ┆ 2.0  ┆ inf ┆ 0.332   ┆ 0.664  ┆ 0.7172  ┆       │
│ abs_target_angle:target_catego… ┆ 5.0  ┆ inf ┆ 0.473   ┆ 2.365  ┆ 0.7969  ┆       │
│ trial_category:target_category  ┆ 10.0 ┆ inf ┆ 1.457   ┆ 14.57  ┆ 0.1487  ┆       │
│ abs_target_angle:trial_categor… ┆ 10.0 ┆ inf ┆ 2.133   ┆ 21.33  ┆ 0.01893 ┆ *     │
└─────────────────────────────────┴──────┴─────┴─────────┴────────┴─────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x00000269665DE540>, _boxhead=Boxhead([ColInfo(var='model term', type=<ColInfoTypeEnum.default: 1>, column_label='model term', column_align='left', column_width=None), ColInfo(var='df1', type=<ColInfoTypeEnum.default: 1>, column_label='df1', column_align='right', column_width=None), ColInfo(var='df2', type=<ColInfoTypeEnum.default: 1>, column_label='df2', column_align='right', column_width=None), ColInfo(var='F_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='F_ratio', column_align='right', column_width=None), ColInfo(var='Chisq', type=<ColInfoTypeEnum.default: 1>, column_label='Chisq', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p_value', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x00000269AF326600>, _spanners=Spanners([]), _heading=Heading(title='ANOVA (Type III tests)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x00000269AF3BCB90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x00000269AF3BF710>, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x00000269AF3BC650>, _formats=[<great_tables._gt_data.FormatInfo object at 0x00000269AF3BC800>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_c

#### Marginal Effect: Trial Categories

In [16]:
model.emmeans("trial_category")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



trial_category,prob,SE,df,asymp_LCL,asymp_UCL
cat,f64,f64,f64,f64,f64
"""BW""",0.659038,0.022752,inf,0.602823,0.711109
"""COLOR""",0.82712,0.01618,inf,0.785021,0.862419
"""NOISE""",0.802049,0.017532,inf,0.756849,0.840617


In [17]:
model.emmeans("trial_category", contrasts="pairwise", p_adjust="tukey")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
cat,f64,f64,f64,f64,f64,f64,f64,f64
"""BW / COLOR""",0.404,0.045683,inf,0.309946,0.526595,1.0,-8.015281,4.3077e-14
"""BW / NOISE""",0.477045,0.052667,inf,0.368286,0.617921,1.0,-6.704062,6.0831e-11
"""COLOR / NOISE""",1.180804,0.143149,inf,0.888758,1.568817,1.0,1.370914,0.35624


#### Margeinal Effect: Target Categories

In [18]:
model.emmeans("target_category")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



target_category,prob,SE,df,asymp_LCL,asymp_UCL
cat,f64,f64,f64,f64,f64
"""ANIMAL_FACE""",0.637798,0.027818,inf,0.56193,0.707369
"""ANIMAL_OTHER""",0.680377,0.026841,inf,0.606055,0.746541
"""HUMAN_FACE""",0.820583,0.020852,inf,0.759085,0.869091
"""HUMAN_OTHER""",0.752558,0.024108,inf,0.683883,0.810448
"""OBJECT_HANDMADE""",0.836409,0.020159,inf,0.776279,0.882818
"""OBJECT_NATURAL""",0.840418,0.019797,inf,0.781251,0.885919


In [19]:
model.emmeans("target_category", contrasts="pairwise", p_adjust="tukey")

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
cat,f64,f64,f64,f64,f64,f64,f64,f64
"""ANIMAL_FACE / ANIMAL_OTHER""",0.827219,0.115479,inf,0.555714,1.231373,1.0,-1.358794,0.751723
"""ANIMAL_FACE / HUMAN_FACE""",0.38501,0.060005,inf,0.246937,0.600285,1.0,-6.124269,1.3649e-8
"""ANIMAL_FACE / HUMAN_OTHER""",0.578984,0.083894,inf,0.383122,0.874974,1.0,-3.771456,0.002244
"""ANIMAL_FACE / OBJECT_HANDMADE""",0.344407,0.055544,inf,0.217508,0.54534,1.0,-6.609411,5.7859e-10
"""ANIMAL_FACE / OBJECT_NATURAL""",0.334364,0.053961,inf,0.2111,0.529602,1.0,-6.788332,1.7015e-10
…,…,…,…,…,…,…,…,…
"""HUMAN_FACE / OBJECT_HANDMADE""",0.89454,0.158717,inf,0.539526,1.483157,1.0,-0.628117,0.988971
"""HUMAN_FACE / OBJECT_NATURAL""",0.868455,0.154066,inf,0.523831,1.439803,1.0,-0.795024,0.968454
"""HUMAN_OTHER / OBJECT_HANDMADE""",0.594847,0.099925,inf,0.368558,0.960073,1.0,-3.092265,0.024337


## M10 Severity Audit

The `pymer4`/`glmer` model above (`is_hit ~ abs_target_angle * trial_category * target_category +
(1 | subject)`) **never converged** (`Convergence status: [1] FALSE`, printed after the fit above) - the
same "ill-conceived" flat random-effects specification `CODE_REVIEW.md` M10 documents for the funnel-based
GAM/GLMM analyses: the unit of observation here is an identification action, and identifications nest
within trial within subject, so `(1|subject)` alone doesn't absorb that structure.

Refit below via plain `lme4::glmer()` (`analysis/R/hit_rate_glmm.R`) as a flat/nested pair, for a clean,
directly-comparable coefficient/AIC/BIC/R² table (the original `pymer4` fit's non-convergence made it an
unreliable baseline to pair against a new nested model). Also switches `abs_target_angle` ->
`centered_abs_target_angle` since we're refitting anyway and every other analysis in this repo already
centers it - mean-centering interaction terms reduces collinearity between main effects and interactions,
which may also help the convergence problem the original model hit.

$$\text{logit}(P[\text{hit}]) \sim \text{centered\_abs\_target\_angle} * C(\text{trial\_category}) * C(\text{target\_category}) + (1 | \text{subject})$$

Fit via `rpy2` (`analysis/helpers/r_bridge.py`), which this session fixed - see `CLAUDE.md` and this
session's M10 severity report for the root-cause diagnosis.

In [20]:
def _fit_hit_rate_glmm():
    model_data = hits_for_angle_regression.copy()
    model_data["centered_abs_target_angle"] = model_data["abs_target_angle"] - model_data["abs_target_angle"].mean()
    to_r_dataframe(
        model_data[["subject", "trial", "trial_category", "target_category", "centered_abs_target_angle", "is_hit"]], "dat",
    )
    source_r(os.path.join(os.getcwd(), "R", "hit_rate_glmm.R"))
    return {
        "metrics_flat": glmer_metrics(get_r_object("model_flat"), "flat (1|subject)"),
        "metrics_nested": glmer_metrics(get_r_object("model_nested"), "nested (1|subject/trial)"),
    }


hit_rate_glmm_result = cached_fit(os.path.join(os.getcwd(), "R", "_cache", "hit_rate_glmm.pkl"), _fit_hit_rate_glmm)

for m in (hit_rate_glmm_result["metrics_flat"], hit_rate_glmm_result["metrics_nested"]):
    print(f"--- {m['label']} --- AIC={m['aic']:.1f}  BIC={m['bic']:.1f}  R2m={m['r2_marginal']:.4f}  R2c={m['r2_conditional']:.4f}  converged={m['converged']}")
    display(m["coefficients"])

--- flat (1|subject) --- AIC=3226.4  BIC=3449.0  R2m=0.1087  R2c=0.1456  converged=True


,term,estimate,std._error,z_value,prz
0,(Intercept),0.935094,0.186015,5.026985,4.982504e-07
1,centered_abs_target_angle,0.036992,0.031139,1.187984,2.348397e-01
2,trial_categoryBW,-0.705400,0.245846,-2.869280,4.114074e-03
3,trial_categoryNOISE,0.166584,0.244694,0.680786,4.960067e-01
4,target_categoryHUMAN_OTHER,0.782268,0.268690,2.911411,3.598000e-03
5,target_categoryANIMAL_FACE,-0.245195,0.235877,-1.039505,2.985701e-01
6,target_categoryHUMAN_FACE,1.087483,0.291055,3.736347,1.867129e-04
7,target_categoryOBJECT_HANDMADE,1.151155,0.313515,3.671774,2.408723e-04
8,target_categoryOBJECT_NATURAL,1.005520,0.289406,3.474431,5.119372e-04
9,centered_abs_target_angle:trial_categoryBW,-0.085024,0.044313,-1.918704,5.502180e-02


--- nested (1|subject/trial) --- AIC=3225.0  BIC=3453.6  R2m=0.1089  R2c=0.1950  converged=True


,term,estimate,std._error,z_value,prz
0,(Intercept),0.993198,0.190922,5.202111,1.970374e-07
1,centered_abs_target_angle,0.039726,0.032032,1.240214,2.148964e-01
2,trial_categoryBW,-0.746108,0.252962,-2.949485,3.183039e-03
3,trial_categoryNOISE,0.173436,0.251645,0.689208,4.906923e-01
4,target_categoryHUMAN_OTHER,0.810345,0.275254,2.943995,3.240048e-03
5,target_categoryANIMAL_FACE,-0.252748,0.241365,-1.047164,2.950240e-01
6,target_categoryHUMAN_FACE,1.092223,0.298569,3.658191,2.540021e-04
7,target_categoryOBJECT_HANDMADE,1.168525,0.321532,3.634242,2.787991e-04
8,target_categoryOBJECT_NATURAL,1.026085,0.297300,3.451347,5.577956e-04
9,centered_abs_target_angle:trial_categoryBW,-0.088529,0.045560,-1.943127,5.200084e-02
